# Adatacore CoreForge — Chatbot LoRA (Research)
This notebook prepares an openly licensed dataset and fine-tunes a **Qwen3-0.6B LoRA adapter**. It does not replace your live Adatacore chatbot, deploy a production model, or consume paid GPUs by itself.

**Before running:** use Google Colab or Kaggle with GPU selected. Free GPU availability is not guaranteed. Inspect the upstream OpenAssistant OASST1 Apache 2.0 terms and samples before training. Never put client or sensitive Adatacore records in this notebook.


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before running training.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
!git clone -q https://github.com/ajay333X/my-adatacore-site.git /content/my-adatacore-site || true
%cd /content/my-adatacore-site
!pip -q install 'unsloth' 'datasets>=3.0,<5' 'transformers>=4.51,<5' 'accelerate>=1.4,<2' 'peft>=0.15,<1'


## 1. Prepare licensed public data
This pipeline redacts common contact identifiers, filters obvious sensitive examples, deduplicates, and creates a reproducible train/validation/test split. It is a prototype, **not a replacement for human review or license verification**.


In [ ]:
!python workers/coreforge_chat_data.py --source oasst1 --limit 500 --accept-license --out /content/coreforge-chatbot-data


## 2. Fine-tune Qwen3-0.6B with LoRA
This step performs real GPU training and evaluates on held-out examples. The output is a LoRA adapter, **not** an always-on chatbot server.


In [ ]:
!python workers/coreforge_chat_train.py --data /content/coreforge-chatbot-data --output /content/coreforge-chatbot-checkpoint --model Qwen/Qwen3-0.6B --max-steps 100 --approve-research-run


In [ ]:
import json
from pathlib import Path
report=json.loads(Path('/content/coreforge-chatbot-checkpoint/training_report.json').read_text())
print(json.dumps(report,indent=2))


## 3. Save your adapter
Colab runtimes can disconnect. You must save the adapter directory to persistent storage (Drive, your own Hugging Face repository, or your own computer). It remains a **research checkpoint** until a real quality benchmark and inference deployment are completed.
